<a href="https://colab.research.google.com/github/TurkuNLP/ATP_kurssi/blob/master/2026/atp_unix_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Working with Text (Unix 2)


## Wildcards (`*`)

Wildcards allow us to work with multiple files without typing each filename separately.

- `*` matches **any number of characters** (including zero).

**Examples:**

```bash
ls *.txt   # Lists all files whose name ends with `.txt` in the current directory.
ls a*      # Lists all files whose names start with `a`.
ls a*.txt  # Lists all files whose name start with `a` and ends with `.txt`.

cp *.txt backup/  # Copies all `.txt` files from the current directory into the `backup` directory.
```


## Searching text with `grep`

The `grep` command searches for lines containing a specified pattern.

**General syntax:**

```bash
grep [options] PATTERN FILENAME
```

**Options:**

```bash
grep "Alice" alice.txt     # Prints all lines in `alice.txt` containing `Alice`.
grep -i "Alice" alice.txt  # Ignore capitalization
grep -c "Alice" alice.txt  # Count matching lines
grep -o "Alice" alice.txt  # Print only the matching parts

grep -w "cat" alice.txt  # Match whole words only (without -w "cat" matches also e.g. catalog and location)
```

**Examples:**

```bash
grep -i "alice" alice.txt  # Prints lines containing `alice`, `Alice`, or `ALICE`, and also `Alices` or `Alice's` (no whole words only option given), prints matching lines
grep -c "Alice" alice.txt  # Counts how many lines contain `Alice`, prints a number
grep -o "Alice" alice.txt  # Prints each occurrence of `Alice` separately, nothing else
grep -iw "alice" alice.txt # Prints lines containing the whole word `alice`, ignoring capitalization

grep -i "Alice" *.txt      # Searches for `Alice` in all `.txt` files in the current directory, prints lines

```


## Combining commands with pipes (`|`)

Many commands produce output, meaning they print something to the terminal (e.g., file contents, file listings, counts).

A **pipe (`|`)** allows us to combine commands by passing the output of one command directly to another.

**General syntax:**

```bash
command1 | command2
```

- `command1` produces some output
- `command2` takes that output and processes it

### Examples

**Example 1: Display only the first 5 matching lines**

```bash
grep -i "Alice" alice.txt | head -5
```

- `grep` prints lines containing `Alice`
- `head -5` keeps only the first five lines and prints them, discarding others

**Example 2: Count occurrences of a word**

```bash
grep -oiw "Alice" alice.txt | wc -l
```

- `grep -oiw` finds each occurrence of the whole word `Alice`, ignoring capitalization, and prints each match separately (each match in a separate line)
- `wc -l` counts the number of lines

**Result:** The total number of occurrences of `Alice` in the file.

**Note:** `grep -c` counts matching **lines**, whereas `grep -o | wc -l` allows us to count **individual occurrences**. `grep -oc` still counts matching lines rather than individual occurrences, which might seem a bit counterintuitive, so we need pipes to get the count of **individual occurrences**.

**Example 3: Find lines containing two words**

```bash
grep -i "Alice" alice.txt | grep -i "Rabbit"
```

- The first `grep` prints only lines containing `Alice`
- The second `grep` takes the input of the first one, and prints only those lines that also contain `Rabbit`

**Result:** Lines containing both words.

**Example 4: More than two commands**

We can also combine more than two commands:

```bash
grep -i "Alice" alice.txt | grep -i "Rabbit" | head -5
```

This prints lines containing both `Alice` and `Rabbit`, but displays only the first five matching lines.



---
## Exercise: Exploring literary texts with Unix commands

In this exercise, we'll use Unix commands to explore and compare two books:

- *Alice's Adventures in Wonderland*
- *The Adventures of Sherlock Holmes*

We'll investigate how certain words are used and how frequently they occur.

### Part 1: Preparing our data

Use commands from Unix 1 Notebook to prepare data. The next step assumes you have these two books downloaded, and named `alice.txt` and `sherlock.txt`.

### Part 2: Exploring the texts

1. How many words in total are there in each book? Which book is longer?
2. Find all lines containing the word `Duchess` in `alice.txt`. Make your search case-insensitive and match whole words only.
3. Since there may be many matching lines, use a pipe to display only the first 5 matching lines.
4. How many lines contain the word `Duchess` in each book? How many individual occurences of the word `Duchess` there is in each book?



---
## Extra: Finding words in context (KWIC)

In corpus linguistics, we often want to find all occurrences of a word and examine the contexts in which it appears.

One common tool for this is a **KWIC (Key Word in Context) concordance**, which displays a word together with some surrounding text.

This is something we can easily achieve using `grep`.

### KWIC using grep (simple version)

This version uses only commands we already know. It will give a limited version of the KWIC interface, which works reasonably well for our example books, but it is not perfect and will not generalize to all kinds of text files.

```bash
grep -iw "Alice" alice.txt
```


**Explanation:**

Why this resembles KWIC concordance:

- KWIC Concordance displays a word (here Alice) together with some surrounding text (limited number of previous and next words).
- `grep` matches lines, so the command returns all lines which contain the word Alice.
- Since this txt version of the book is divided into lines that matches the original lines in the book page, the context we get is the surrounding line (if the txt file would have another notion of lines, we would get something else)

Limitations:

- KWIC Concordance displays the search word in the middle, while this just prints the line, with the word appearning somewhere there
- Preceeding or following context may be missing because of the line break
- Context length varies

A proper KWIC concordance typically displays a fixed amount of context around each occurrence, with the keywords aligned in the middle.

### Improved KWIC using grep (full version)

**You don't need to undertand the command, this is to show you what can be done rather than requiring you to learn this.**

We can improve our concordance using additional Unix commands for text normalization (here converting newlines to spaces) and regular expressions. Text normalization and regular expressions will be discussed in upcoming weeks.

```bash
cat alice.txt | perl -pe 's/\r\n/ /g' | grep -oiP '.{0,40}Alice.{0,40}'
```

**What happens here?**

- `cat` read the file content and prints it
- `perl` replaces each occurrance of new line with whitespace. This converts the individual lines into one long text
- `grep -o` searches individual occurences of "Alice" from this text, printing one occurence per line
- The regular expression `.{0,40}Alice.{0,40}` add preceeding and following context to the occurence of Alice, it reads like "include any 40 characters before and after Alice"

This will now display us something which is really close to typical KWIC Concordance list.


**Limitations:** This is still not a perfect KWIC concordance: nearby occurrences can be grouped into one match, and in that case keywords are not always aligned.
